# Task 2: using the trained genre classifier

Training saved only the small pieces you trained, in the `best/` folder:

| File | What it is |
|---|---|
| `adapter_config.json`, `adapter_model.safetensors` | the LoRA adapter (tens of MB), plus the name of the base model it belongs to |
| `head.pt` | the 100-genre classification layer |
| `label2id.json` | which output number means which genre |

The 4B base model is **not** in that folder; it is downloaded from Hugging Face as usual. Using the classifier means rebuilding the same stack as in training: **base model → attach adapter → attach head**. Then the one loaded model does both jobs:

- **classify** with the adapter on (the default), and
- **chat** inside `with model.disable_adapter():`, which is exactly the original chatbot.

**Where to run it.** On Colab with a T4 (recommended): the base loads in 4-bit, the same way it was trained. On a machine without an NVIDIA GPU, 4-bit loading is not available, so the notebook falls back to loading the base in bfloat16 on the CPU. That needs about 9 GB of free RAM and takes several seconds per story, and predictions can differ very slightly from the 4-bit numbers. Task 3 is where the fast CPU version comes from.

## 1. Install and point at the trained files

In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes

In [ ]:
# On Colab: mount Drive, where the training notebook saved best/
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import json

# The best/ folder written by the training notebook. On your own machine, download that folder from Drive
# and put its local path here, e.g. Path("models/task2_best").
ADAPTER_DIR = Path("/content/drive/MyDrive/story-chatbot/task2/Qwen3-4B-Instruct-2507/best")

assert (ADAPTER_DIR / "adapter_config.json").exists(), f"no adapter in {ADAPTER_DIR}; check the path"
MODEL_ID = json.load(open(ADAPTER_DIR / "adapter_config.json"))["base_model_name_or_path"]   # always matches the adapter
label2id = json.load(open(ADAPTER_DIR / "label2id.json"))
id2label = {i: g for g, i in label2id.items()}

# Input settings must match training. The training notebook stores them in results.json next to best/.
settings = {"max_len": 1024, "use_title": True}
results_path = ADAPTER_DIR.parent / "results.json"
if results_path.exists():
    settings.update({k: v for k, v in json.load(open(results_path))["hyperparameters"].items() if k in settings})
else:
    print("results.json not found; using MAX_LEN=1024 and USE_TITLE=True. Change them if you trained differently.")
MAX_LEN, USE_TITLE = settings["max_len"], settings["use_title"]
print(f"base model: {MODEL_ID} | {len(label2id)} genres | MAX_LEN={MAX_LEN} | USE_TITLE={USE_TITLE}")

## 2. Load base model, adapter and head

`PeftModel.from_pretrained` wraps the base model and inserts the adapter layers; it does not change any base weight. The head is rebuilt with the same class as in training and its saved weights are loaded into it.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base_model():
    if DEVICE.type == "cuda":     # same 4-bit setup as training and the chatbot
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                 bnb_4bit_compute_dtype=torch.float16)
        return AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, dtype=torch.float16, device_map={"": 0})
    print("No GPU: loading the base in bfloat16 on CPU (slow, ~9 GB RAM)")
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16)

base = load_base_model()
base.config.pad_token_id = tokenizer.pad_token_id
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()                                 # inference mode (no dropout)
print("adapter attached:", list(model.peft_config))

In [ ]:
import torch.nn as nn

class GenreClassifier(nn.Module):           # identical to the training notebook
    def __init__(self, peft_model, num_labels, dropout=0.1):
        super().__init__()
        self.peft_model = peft_model
        hidden = peft_model.get_base_model().config.hidden_size
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, num_labels))
        self.head.to(next(peft_model.parameters()).device, dtype=torch.float32)

    def forward(self, input_ids, attention_mask):
        transformer = self.peft_model.get_base_model().model
        h = transformer(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).last_hidden_state
        last = attention_mask.sum(dim=1) - 1
        pooled = h[torch.arange(h.size(0), device=h.device), last]
        return self.head(pooled.float())

clf = GenreClassifier(model, len(label2id))
clf.head.load_state_dict(torch.load(ADAPTER_DIR / "head.pt", map_location=clf.head[1].weight.device))
clf.eval()                                   # turns dropout off: predictions are deterministic
if torch.cuda.is_available():
    print(f"GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.2f} GB")

## 3. Predict genres

The input must be built **exactly** as in training: `Title: …`, the story truncated to fit, then the suffix `The genre of this story is:`. A different format would give the head hidden states it never saw, and accuracy would drop without any error message.

`predict_genre` returns the top-k genres with their probabilities. `predict_many` does a list of stories in batches (padding on the right, like training).

In [ ]:
from contextlib import nullcontext

SUFFIX = "\n\nThe genre of this story is:"
suffix_ids = tokenizer(SUFFIX, add_special_tokens=False)["input_ids"]

def encode(title, story):
    text = f"Title: {title}\n\n{story}" if USE_TITLE else story
    body = tokenizer(text, add_special_tokens=False)["input_ids"][: MAX_LEN - len(suffix_ids)]
    return body + suffix_ids

def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if DEVICE.type == "cuda" else nullcontext()

@torch.no_grad()
def predict_many(items, k=3, batch_size=4):
    # items: list of (title, story). Returns one list of (genre, probability) per story.
    out = []
    for s in range(0, len(items), batch_size):
        encoded = [encode(t, st) for t, st in items[s : s + batch_size]]
        width = max(map(len, encoded))
        ids = torch.full((len(encoded), width), tokenizer.pad_token_id, dtype=torch.long)
        mask = torch.zeros_like(ids)
        for i, e in enumerate(encoded):
            ids[i, : len(e)] = torch.tensor(e); mask[i, : len(e)] = 1
        with autocast():
            probs = clf(ids.to(DEVICE), mask.to(DEVICE)).float().softmax(-1)
        top = probs.topk(k)
        out += [[(id2label[i], round(p, 3)) for p, i in zip(pv.tolist(), iv.tolist())] for pv, iv in zip(top.values, top.indices)]
    return out

def predict_genre(title, story, k=3):
    return predict_many([(title, story)], k=k)[0]

predict_genre("The Last Signal",
              "The colony ship drifted past Neptune when the AI woke the crew. A message was coming from a star "
              "that had died a thousand years ago, and it was addressed to the captain by name.")

### Classify a whole file

For a CSV with `title` and `story` columns (for example the dataset's test split or your own stories), this adds the predicted genre and its probability.

In [ ]:
import pandas as pd

def classify_frame(df, k=3):
    preds = predict_many(list(zip(df["title"].astype(str), df["story"].astype(str))), k=k)
    df = df.copy()
    df["pred_genre"] = [p[0][0] for p in preds]
    df["pred_prob"] = [p[0][1] for p in preds]
    df[f"top{k}"] = [", ".join(g for g, _ in p) for p in preds]
    return df

# Example: df = pd.read_csv("my_stories.csv"); classify_frame(df).to_csv("my_stories_with_genres.csv", index=False)

### Optional: check the reloaded model on the test split

A quick way to confirm everything loaded correctly: the accuracy here should match the test accuracy printed at the end of training. Skip it if you only want predictions.

In [ ]:
SPLITS_PATH = Path("/content/drive/MyDrive/story-chatbot/data/splits.json")    # the split used in training
RUN_TEST_CHECK = SPLITS_PATH.exists()

if RUN_TEST_CHECK:
    from datasets import load_dataset
    raw = load_dataset("FareedKhan/1k_stories_100_genre")
    df_all = raw[list(raw.keys())[0]].to_pandas()
    df_all["id"] = df_all["id"].astype(str)
    test_ids = set(json.load(open(SPLITS_PATH))["test"])
    test = classify_frame(df_all[df_all["id"].isin(test_ids)])
    print(f"test accuracy: {(test['pred_genre'] == test['genre'].str.strip()).mean():.3f} on {len(test)} stories")

## 4. Chat with the same model (adapter off)

The chatbot path never uses the adapter or the head. Any generation call wrapped in `disable_adapter()` runs on the untouched base weights, so the Task 1 chatbot can share this one loaded model. In the real app, your Task 1 pipeline builds the prompt from retrieved stories; here we pass one story directly to show the switch.

In [ ]:
@torch.no_grad()
def chat(question, story_text, max_new_tokens=200):
    messages = [{"role": "system", "content": "Answer only from the story provided. If the answer is not in the story, say so."},
                {"role": "user", "content": f"<story>\n{story_text}\n</story>\n\nQuestion: {question}"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with model.disable_adapter():                     # chatbot = original base model
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                             top_k=None, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

story = ("The colony ship drifted past Neptune when the AI woke the crew. A message was coming from a star "
         "that had died a thousand years ago, and it was addressed to the captain by name.")
print(chat("Who was the message addressed to?", story))
print("genre:", predict_genre("The Last Signal", story, k=1))

## Moving this into the codebase

When you add the `classify/` part of the project, the same pieces become a small module: a `load_classifier(adapter_dir, base_model)` function (section 2) and `predict_many` (section 3). The chatbot and the classifier should then share the one `PeftModel`, so the 4B weights are loaded only once; that is what keeps both tasks inside the memory budget.